# 8. Despliegue (MLOps): API, contenedor, pruebas e integración continua

**Objetivo del capítulo.** Dejar el modelo listo para usarse fuera del notebook y comprobar, desde aquí mismo,
que la API responde como se especifica.

| Pieza | Archivo | Qué hace |
|---|---|---|
| API | `app/api.py`, `app/schemas.py` | FastAPI: `POST /predict` recibe `lags` y `horizon ∈ {7, 14, 21, 28}` y devuelve H predicciones; `GET /info` informa cuántos lags espera cada horizonte; `GET /health`. |
| Modelos | `app/model_h{H}.keras`, `app/scalers_h{H}.joblib`, `app/model_card.json` | Mejor MLP por horizonte (capítulo 5) y sus escaladores ajustados solo con entrenamiento. |
| Contenedor | `Dockerfile` | Imagen `python:3.12-slim` con TensorFlow CPU; sirve la API con uvicorn en el puerto 8000. |
| Pruebas | `tests/` | Forma de la salida de cada modelo, coherencia modelo–escaladores, API (códigos 200/422), ausencia de fuga en la validación y métricas. |
| CI | `.github/workflows/ci.yml` | En cada *push*: flake8 + pytest. |

In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from IPython.display import Markdown, display

from fcst.nbutils import setup

cfg, RES, FIG = setup()
APP = cfg.path("app_dir")
os.environ["MODEL_DIR"] = str(APP)
from fastapi.testclient import TestClient  # noqa: E402

from app.api import app  # noqa: E402

client = TestClient(app)
card = json.loads((APP / "model_card.json").read_text(encoding="utf8"))

Perfil: full  |  resultados en: /content/deep-learning-02-forecasting-mlp/results


## 8.1 La API en funcionamiento

In [2]:
print(client.get("/health").json())
pd.DataFrame(client.get("/info").json())

{'status': 'ok', 'horizons': [7, 14, 21, 28]}


,horizon,n_lags,arquitectura,dropout_mask,rmse_validacion_usd,rmse_holdout_usd
0,7,21,"(64, 128, 256)","(False, False, True)",4035.175287,5343.430693
1,14,7,"(256, 256)","(False, True)",6647.052940,4822.924162
2,21,7,"(256, 256)","(True, True)",6337.056343,5901.038855
3,28,7,"(256, 256, 64)","(False, False, False)",7491.821846,7046.898650


Pronóstico con los **últimos cierres conocidos** de la serie (los del final del hold-out), para cada horizonte:

In [3]:
last = card["ultimos_cierres"]
rows = []
for H in [7, 14, 21, 28]:
    n = {d["horizon"]: d["n_lags"] for d in client.get("/info").json()}[H]
    r = client.post("/predict", json={"lags": last[-n:], "horizon": H})
    p = r.json()["prediction"]
    rows.append({"H": H, "lags usados": n, "estado HTTP": r.status_code, "día +1": p[0], f"día +H": p[-1],
                 "mín": min(p), "máx": max(p)})
display(Markdown(f"Último cierre conocido: **{card['ultimo_cierre_conocido']['close']:,.0f} USD** "
                 f"({card['ultimo_cierre_conocido']['fecha']})."))
pd.DataFrame(rows).style.format({c: "{:,.0f}" for c in ["día +1", "día +H", "mín", "máx"]}).hide(axis="index")

Último cierre conocido: **77,582 USD** (2026-10-31).

H,lags usados,estado HTTP,día +1,día +H,mín,máx
7,21,200,"82,375","80,707","80,707","82,375"
14,7,200,"77,369","78,387","76,983","78,572"
21,7,200,"77,419","80,017","77,367","80,271"
28,7,200,"71,564","80,497","71,564","80,497"


In [4]:
bad = client.post("/predict", json={"lags": [1.0] * 7, "horizon": 10})
print("Horizonte inválido ->", bad.status_code, bad.json()["detail"][0]["msg"])
bad2 = client.post("/predict", json={"lags": [1.0] * 3, "horizon": 7})
print("Número de lags incorrecto ->", bad2.status_code, bad2.json()["detail"])

Horizonte inválido -> 422 Input should be 7, 14, 21 or 28
Número de lags incorrecto -> 422 Se esperaban 21 lags para H=7, se recibieron 3


## 8.2 Pruebas unitarias

In [5]:
out = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], cwd=ROOT, capture_output=True, text=True,
                     env={**os.environ, "MODEL_DIR": str(APP), "PYTHONPATH": f"{ROOT}:{ROOT / 'src'}"})
print(out.stdout[-1500:])
assert out.returncode == 0, out.stderr[-2000:]

.....................................                                    [100%]
37 passed in 10.30s



## 8.3 Uso local y con Docker

```bash
# Local
pip install -r requirements-api.txt
uvicorn app.api:app --port 8000

# Docker
docker build -t btc-forecast-mlp .
docker run -p 8000:8000 btc-forecast-mlp

# Consulta (horizonte de 14 días; envíe tantos lags como indique /info para ese horizonte)
curl -X POST http://localhost:8000/predict -H "Content-Type: application/json" \
     -d '{"lags": [...], "horizon": 14}'
```

La documentación interactiva queda en `http://localhost:8000/docs`. El modelo es un ejercicio académico de
pronóstico: no constituye una recomendación de inversión.